# 🏥 SOTA Medical Image Segmentation 5-Fold Cross-Validation Benchmark (60 Epochs)
# Models: UNet++ (Nested SOTA), U-Net, Attention U-Net
# Hardware: Dual-GPU Tesla T4 x2 (Kaggle Cloud)

In [ ]:
# 1. Environment & Hardware Detection (Dual-GPU T4 x2)
import os, sys, glob, json, time, random, shutil, zipfile

# Restrict background CPU thread contention to keep CPU cool and prevent starvation
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import cv2
cv2.setNumThreads(0)

import matplotlib.pyplot as plt
from scipy.spatial.distance import directed_hausdorff

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
torch.set_num_threads(2)

print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")
    print(f"Device Count   : {torch.cuda.device_count()}")
print("=" * 60)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)

In [ ]:
# 2. Ingest Dataset & Setup 5-Fold Stratified Split
DATA_DIR = "/kaggle/working/data/kvasir_seg"
os.makedirs(os.path.join(DATA_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, "masks"), exist_ok=True)

# Scan for Kaggle uploaded dataset
input_imgs = glob.glob("/kaggle/input/**/kvasir*/**/images/*.*", recursive=True)
if not input_imgs:
    input_imgs = glob.glob("/kaggle/input/**/images/*.*", recursive=True)

if input_imgs:
    print(f"[MODE: MOUNTED DATASET] Ingesting {len(input_imgs)} cases from Kaggle input...")
    img_dir_found = os.path.dirname(input_imgs[0])
    parent_dir = os.path.dirname(img_dir_found)
    mask_dir_found = os.path.join(parent_dir, "masks")
    
    # Check if pre-split exists
    if os.path.exists(os.path.join(parent_dir, "splits.json")):
        shutil.copy(os.path.join(parent_dir, "splits.json"), os.path.join(DATA_DIR, "splits.json"))
        print("[SUCCESS] Loaded pre-computed stratified splits.json directly!")
        
    for p in glob.glob(os.path.join(img_dir_found, "*.*")):
        shutil.copy(p, os.path.join(DATA_DIR, "images", os.path.basename(p)))
    for p in glob.glob(os.path.join(mask_dir_found, "*.*")):
        shutil.copy(p, os.path.join(DATA_DIR, "masks", os.path.basename(p)))
else:
    print("[MODE: DIRECT DOWNLOAD] Downloading official Simula Kvasir-SEG...")
    import urllib.request
    url = "https://datasets.simula.no/downloads/kvasir-seg.zip"
    zip_path = "/kaggle/working/kvasir_seg.zip"
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall("/kaggle/working/data")
    if os.path.exists("/kaggle/working/data/Kvasir-SEG"):
        os.rename("/kaggle/working/data/Kvasir-SEG", DATA_DIR)

print(f"Dataset ready: {len(os.listdir(os.path.join(DATA_DIR, 'images')))} images, {len(os.listdir(os.path.join(DATA_DIR, 'masks')))} masks!")

# Generate 5-Fold Stratified Split if not present
if not os.path.exists(os.path.join(DATA_DIR, "splits.json")):
    img_files = sorted(glob.glob(os.path.join(DATA_DIR, "images", "*.*")))
    stems, ratios = [], []
    for ip in img_files:
        stem = os.path.splitext(os.path.basename(ip))[0]
        mp = glob.glob(os.path.join(DATA_DIR, "masks", f"{stem}.*"))[0]
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        ratios.append(float((m > 127).sum()) / (m.shape[0] * m.shape[1]))
        stems.append(stem)
    
    stems, ratios = np.array(stems), np.array(ratios)
    bins = np.percentile(ratios, [20, 40, 60, 80])
    strata = np.digitize(ratios, bins)
    rng = np.random.RandomState(42)
    fold_idx = [[] for _ in range(5)]
    for s in range(5):
        s_idx = np.where(strata == s)[0]
        rng.shuffle(s_idx)
        for f, chunk in enumerate(np.array_split(s_idx, 5)):
            fold_idx[f].extend(chunk.tolist())
            
    splits = {}
    for f in range(5):
        v = fold_idx[f]
        t = [i for k in range(5) if k != f for i in fold_idx[k]]
        splits[f"fold_{f}"] = {"train": sorted(stems[t].tolist()), "val": sorted(stems[v].tolist())}
    with open(os.path.join(DATA_DIR, "splits.json"), "w") as fp:
        json.dump(splits, fp, indent=2)

GLOBAL_CACHE = {}

def preload_kvasir_to_ram(data_dir, img_size=256):
    global GLOBAL_CACHE
    if len(GLOBAL_CACHE) > 0:
        return
    print(f"[*] Preloading entire dataset into CPU RAM ({img_size}x{img_size}) to eliminate disk I/O and drop CPU load...")
    img_dir = os.path.join(data_dir, "images")
    mask_dir = os.path.join(data_dir, "masks")
    files = sorted(glob.glob(os.path.join(img_dir, "*.*")))
    for ip in files:
        stem = os.path.splitext(os.path.basename(ip))[0]
        mp_cands = glob.glob(os.path.join(mask_dir, f"{stem}.*"))
        if not mp_cands: continue
        img = cv2.imread(ip)
        if img is None: continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = (cv2.imread(mp_cands[0], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
        GLOBAL_CACHE[stem] = (img, mask)
    print(f"[SUCCESS] Preloaded {len(GLOBAL_CACHE)} pairs into RAM (~250 MB total)! CPU disk I/O eliminated!")

class KvasirDataset(Dataset):
    def __init__(self, data_dir, split="train", fold=0, img_size=256):
        self.split = split
        with open(os.path.join(data_dir, "splits.json")) as fp:
            self.stems = json.load(fp)[f"fold_{fold}"][split]
        preload_kvasir_to_ram(data_dir, img_size)

    def __len__(self): return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        img, mask = GLOBAL_CACHE[stem]
        img = img.copy()
        mask = mask.copy()
        
        if self.split == "train":
            if random.random() > 0.5: img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
            if random.random() > 0.5: img, mask = np.flipud(img).copy(), np.flipud(mask).copy()
            if random.random() > 0.5:
                k = random.randint(1, 3)
                img, mask = np.rot90(img, k).copy(), np.rot90(mask, k).copy()
                
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).float().unsqueeze(0)
        return img, mask

def get_dataloaders(fold=0, batch_size=16, img_size=256):
    td = KvasirDataset(DATA_DIR, split="train", fold=fold, img_size=img_size)
    vd = KvasirDataset(DATA_DIR, split="val", fold=fold, img_size=img_size)
    # num_workers=0 loads directly from in-memory RAM cache: 0% worker overhead, 0% /dev/shm shared memory, CPU load drops to <10%!
    tl = DataLoader(td, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
    vl = DataLoader(vd, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
    return tl, vl

In [ ]:
# 3. Standard Medical Losses: CompoundLoss with Boundary-IoU
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    def forward(self, pred, target):
        p = torch.sigmoid(pred).view(-1)
        t = target.view(-1)
        inter = (p * t).sum()
        return 1.0 - ((2.0 * inter + self.smooth) / (p.sum() + t.sum() + self.smooth))

class BoundaryIoULoss(nn.Module):
    def __init__(self, d=3, smooth=1e-5):
        super().__init__()
        self.d = d
        self.smooth = smooth
    def extract_boundary(self, x):
        k = 2 * self.d + 1
        dilation = F.max_pool2d(x, kernel_size=k, stride=1, padding=self.d)
        erosion = -F.max_pool2d(-x, kernel_size=k, stride=1, padding=self.d)
        return dilation - erosion
    def forward(self, pred, target):
        p = torch.sigmoid(pred)
        p_b = self.extract_boundary(p).view(-1)
        t_b = self.extract_boundary(target).view(-1)
        inter = (p_b * t_b).sum()
        return 1.0 - ((2.0 * inter + self.smooth) / (p_b.sum() + t_b.sum() + self.smooth))

class CompoundLoss(nn.Module):
    def __init__(self, lambda_bce=1.0, lambda_dice=1.0, lambda_boundary=0.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.boundary = BoundaryIoULoss() if lambda_boundary > 0 else None
        self.lambda_bce = lambda_bce
        self.lambda_dice = lambda_dice
        self.lambda_boundary = lambda_boundary
        
    def forward(self, pred_seg, pred_edge, target):
        loss = self.lambda_bce * self.bce(pred_seg, target) + self.lambda_dice * self.dice(pred_seg, target)
        if self.boundary is not None and self.lambda_boundary > 0:
            loss = loss + self.lambda_boundary * self.boundary(pred_seg, target)
            if pred_edge is not None:
                gt_edge = self.boundary.extract_boundary(target)
                loss = loss + self.lambda_boundary * self.bce(pred_edge, gt_edge)
        return loss

def compute_hd95(p_mask: np.ndarray, t_mask: np.ndarray) -> float:
    if p_mask.sum() == 0 and t_mask.sum() == 0: return 0.0
    if p_mask.sum() == 0 or t_mask.sum() == 0: return 50.0
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    p_b = cv2.morphologyEx(p_mask.astype(np.uint8), cv2.MORPH_GRADIENT, k)
    t_b = cv2.morphologyEx(t_mask.astype(np.uint8), cv2.MORPH_GRADIENT, k)
    p_pts = np.argwhere(p_b > 0)
    t_pts = np.argwhere(t_b > 0)
    if len(p_pts) == 0 or len(t_pts) == 0: return 50.0
    if len(p_pts) > 150: p_pts = p_pts[::len(p_pts)//150]
    if len(t_pts) > 150: t_pts = t_pts[::len(t_pts)//150]
    d1 = directed_hausdorff(p_pts, t_pts)[0]
    d2 = directed_hausdorff(t_pts, p_pts)[0]
    return float(max(d1, d2))

In [ ]:
# 4. Universal 60-Epoch Multi-GPU Training Engine (Anti-OOM & Memory Optimized)
import gc

def train_and_eval_fold(model_fn, fold=0, epochs=60, lr=3e-4, patience=12, lambda_boundary=0.0, save_name="model"):
    gc.collect()
    torch.cuda.empty_cache()
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model_fn()
    n_gpus = torch.cuda.device_count()
    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # Batch size 16 (8 per GPU on Dual T4). Safe for UNet++ without VRAM exhaustion!
    batch_size = 16 if n_gpus > 1 else 8
    tl, vl = get_dataloaders(fold=fold, batch_size=batch_size, img_size=256)
    
    criterion = CompoundLoss(lambda_bce=1.0, lambda_dice=1.0, lambda_boundary=lambda_boundary).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda')
    
    accum_steps = 2 # Gradient accumulation: mathematically identical to batch size 32
    
    best_dice = -1.0
    patience_cnt = 0
    best_metrics = {}
    history = {"epochs": [], "train_loss": [], "val_dice": [], "val_iou": [], "val_hd95": []}
    
    print(f"\n--- Training [{save_name}] Fold {fold} for {epochs} Epochs (Batch: {batch_size}, Accum: {accum_steps}) ---")
    for ep in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad(set_to_none=True)
        
        for step_idx, (imgs, msks) in enumerate(tl):
            imgs, msks = imgs.to(device, non_blocking=True), msks.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                out = model(imgs)
                if isinstance(out, (list, tuple)):
                    seg = out[0]
                    edge = out[1] if len(out) > 1 and out[1] is not None and not isinstance(out[1], list) else None
                else:
                    seg, edge = out, None
                loss = criterion(seg, edge, msks)
                loss = loss / accum_steps
                
            scaler.scale(loss).backward()
            
            if (step_idx + 1) % accum_steps == 0 or (step_idx + 1) == len(tl):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                
            train_loss += loss.item() * accum_steps * imgs.size(0)
            
        scheduler.step()
        train_loss /= len(tl.dataset)
        
        torch.cuda.empty_cache()
        
        # Validation fully accelerated on GPU!
        model.eval()
        dices, ious, hd95s = [], [], []
        with torch.no_grad():
            for imgs, msks in vl:
                imgs, msks = imgs.to(device, non_blocking=True), msks.to(device, non_blocking=True)
                with torch.amp.autocast('cuda'):
                    out = model(imgs)
                    seg = out[0] if isinstance(out, (list, tuple)) else out
                preds = (torch.sigmoid(seg) > 0.5).float()
                
                # Fast GPU Vectorized Dice & IoU (Zero CPU load!)
                inter = (preds * msks).sum(dim=(2, 3))
                total = preds.sum(dim=(2, 3)) + msks.sum(dim=(2, 3))
                union = total - inter
                batch_dice = ((2.0 * inter + 1e-5) / (total + 1e-5)).view(-1).cpu().tolist()
                batch_iou = ((inter + 1e-5) / (union + 1e-5)).view(-1).cpu().tolist()
                dices.extend(batch_dice)
                ious.extend(batch_iou)
                
                # Fast boundary HD95 (0.8ms per image, sub-second total!)
                preds_np = preds.squeeze(1).cpu().numpy().astype(np.uint8)
                msks_np = msks.squeeze(1).cpu().numpy().astype(np.uint8)
                for b in range(preds_np.shape[0]):
                    hd95s.append(compute_hd95(preds_np[b], msks_np[b]))
                    
        val_dice = float(np.mean(dices))
        val_iou = float(np.mean(ious))
        val_hd95 = float(np.mean(hd95s))
        
        history["epochs"].append(ep)
        history["train_loss"].append(float(train_loss))
        history["val_dice"].append(float(val_dice * 100))
        history["val_iou"].append(float(val_iou * 100))
        history["val_hd95"].append(float(val_hd95))
        
        if ep % 5 == 0 or ep == epochs or val_dice > best_dice:
            print(f"Epoch [{ep:02d}/{epochs}] | Train Loss: {train_loss:.4f} | Val Dice: {val_dice*100:.2f}%, IoU: {val_iou*100:.2f}%, HD95: {val_hd95:.2f}")
            
        if val_dice > best_dice:
            best_dice = val_dice
            patience_cnt = 0
            best_metrics = {"dice": val_dice, "iou": val_iou, "hd95": val_hd95, "best_epoch": ep}
            model_to_save = model.module if hasattr(model, 'module') else model
            ckpt_path = f"/kaggle/working/{save_name}_fold{fold}_best.pth"
            torch.save(model_to_save.state_dict(), ckpt_path)
        else:
            patience_cnt += 1
            if patience_cnt >= patience and ep >= 30:
                print(f"Early stopped at epoch {ep}! Best Val Dice: {best_dice*100:.2f}% (Epoch {best_metrics.get('best_epoch')})")
                break
                
    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    
    return best_metrics, history

In [ ]:
# 5. Baseline Architectures: U-Net, Attention U-Net, UNet++
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_c=3, num_classes=1, base_c=32):
        super().__init__()
        c = base_c
        self.inc = DoubleConv(in_c, c)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c, c*2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c*2, c*4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c*4, c*8))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c*8, 512))
        
        self.up1 = nn.ConvTranspose2d(512, c*8, 2, stride=2)
        self.conv1 = DoubleConv(512, c*8)
        self.up2 = nn.ConvTranspose2d(c*8, c*4, 2, stride=2)
        self.conv2 = DoubleConv(c*8, c*4)
        self.up3 = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.conv3 = DoubleConv(c*4, c*2)
        self.up4 = nn.ConvTranspose2d(c*2, c, 2, stride=2)
        self.conv4 = DoubleConv(c*2, c)
        self.outc = nn.Conv2d(c, num_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.conv1(torch.cat([self.up1(x5), x4], dim=1))
        x = self.conv2(torch.cat([self.up2(x), x3], dim=1))
        x = self.conv3(torch.cat([self.up3(x), x2], dim=1))
        x = self.conv4(torch.cat([self.up4(x), x1], dim=1))
        return self.outc(x)

class AttentionGate(nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.w_g = nn.Sequential(nn.Conv2d(f_g, f_int, 1, bias=False), nn.BatchNorm2d(f_int))
        self.w_x = nn.Sequential(nn.Conv2d(f_l, f_int, 1, bias=False), nn.BatchNorm2d(f_int))
        self.psi = nn.Sequential(nn.Conv2d(f_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        return x * self.psi(self.relu(self.w_g(g) + self.w_x(x)))

class AttentionUNet(nn.Module):
    def __init__(self, in_c=3, num_classes=1, base_c=32):
        super().__init__()
        c = base_c
        self.inc = DoubleConv(in_c, c)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c, c*2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c*2, c*4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c*4, c*8))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(c*8, 512))
        
        self.up1 = nn.ConvTranspose2d(512, c*8, 2, stride=2)
        self.ag1 = AttentionGate(f_g=c*8, f_l=c*8, f_int=c*4)
        self.conv1 = DoubleConv(512, c*8)
        
        self.up2 = nn.ConvTranspose2d(c*8, c*4, 2, stride=2)
        self.ag2 = AttentionGate(f_g=c*4, f_l=c*4, f_int=c*2)
        self.conv2 = DoubleConv(c*8, c*4)
        
        self.up3 = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.ag3 = AttentionGate(f_g=c*2, f_l=c*2, f_int=c)
        self.conv3 = DoubleConv(c*4, c*2)
        
        self.up4 = nn.ConvTranspose2d(c*2, c, 2, stride=2)
        self.ag4 = AttentionGate(f_g=c, f_l=c, f_int=c//2)
        self.conv4 = DoubleConv(c*2, c)
        self.outc = nn.Conv2d(c, num_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        u1 = self.up1(x5)
        x = self.conv1(torch.cat([u1, self.ag1(u1, x4)], dim=1))
        u2 = self.up2(x)
        x = self.conv2(torch.cat([u2, self.ag2(u2, x3)], dim=1))
        u3 = self.up3(x)
        x = self.conv3(torch.cat([u3, self.ag3(u3, x2)], dim=1))
        u4 = self.up4(x)
        x = self.conv4(torch.cat([u4, self.ag4(u4, x1)], dim=1))
        return self.outc(x)

class UNetPlusPlus(nn.Module):
    def __init__(self, in_c=3, num_classes=1, base_c=32):
        super().__init__()
        nb = [base_c, base_c*2, base_c*4, base_c*8, 512]
        self.pool = nn.MaxPool2d(2, 2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.conv0_0 = DoubleConv(in_c, nb[0])
        self.conv1_0 = DoubleConv(nb[0], nb[1])
        self.conv2_0 = DoubleConv(nb[1], nb[2])
        self.conv3_0 = DoubleConv(nb[2], nb[3])
        self.conv4_0 = DoubleConv(nb[3], nb[4])

        self.conv0_1 = DoubleConv(nb[0] + nb[1], nb[0])
        self.conv1_1 = DoubleConv(nb[1] + nb[2], nb[1])
        self.conv2_1 = DoubleConv(nb[2] + nb[3], nb[2])
        self.conv3_1 = DoubleConv(nb[3] + nb[4], nb[3])

        self.conv0_2 = DoubleConv(nb[0]*2 + nb[1], nb[0])
        self.conv1_2 = DoubleConv(nb[1]*2 + nb[2], nb[1])
        self.conv2_2 = DoubleConv(nb[2]*2 + nb[3], nb[2])

        self.conv0_3 = DoubleConv(nb[0]*3 + nb[1], nb[0])
        self.conv1_3 = DoubleConv(nb[1]*3 + nb[2], nb[1])

        self.conv0_4 = DoubleConv(nb[0]*4 + nb[1], nb[0])
        self.final = nn.Conv2d(nb[0], num_classes, kernel_size=1)

    def forward(self, x):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0))
        x0_1 = self.conv0_1(torch.cat([x0_0, self.up(x1_0)], 1))

        x2_0 = self.conv2_0(self.pool(x1_0))
        x1_1 = self.conv1_1(torch.cat([x1_0, self.up(x2_0)], 1))
        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self.up(x1_1)], 1))

        x3_0 = self.conv3_0(self.pool(x2_0))
        x2_1 = self.conv2_1(torch.cat([x2_0, self.up(x3_0)], 1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self.up(x2_1)], 1))
        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self.up(x1_2)], 1))

        x4_0 = self.conv4_0(self.pool(x3_0))
        x3_1 = self.conv3_1(torch.cat([x3_0, self.up(x4_0)], 1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self.up(x3_1)], 1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self.up(x2_2)], 1))
        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self.up(x1_3)], 1))
        return self.final(x0_4)

print("Baseline architectures loaded: UNet (7.76M), AttentionUNet (7.85M), UNetPlusPlus (9.16M)")

In [ ]:
# 6. Execute 5-Fold Benchmark for Baselines (UNet++, U-Net, Attention U-Net)
EPOCHS = 60
PATIENCE = 12

MODELS_CONFIG = [
    {"name": "unet_plus_plus", "class": lambda: UNetPlusPlus(in_c=3, num_classes=1, base_c=32), "lambda_boundary": 0.0},
    {"name": "unet", "class": lambda: UNet(in_c=3, num_classes=1, base_c=32), "lambda_boundary": 0.0},
    {"name": "attention_unet", "class": lambda: AttentionUNet(in_c=3, num_classes=1, base_c=32), "lambda_boundary": 0.0}
]

print("==========================================================================")
print(" STARTING RIGOROUS 5-FOLD BASELINE BENCHMARK (60 Epochs per fold)")
print("==========================================================================")

all_benchmark_results = {}
all_benchmark_histories = {}

for cfg in MODELS_CONFIG:
    m_name = cfg["name"]
    print(f"\n######################################################################")
    print(f" >>> BENCHMARKING MODEL: {m_name.upper()} (All 5 Folds, {EPOCHS} Epochs)")
    print(f"######################################################################")
    
    m_fold_metrics = []
    m_fold_histories = []
    
    for f in range(5):
        seed_everything(42 + f)
        best_res, hist = train_and_eval_fold(
            model_fn=cfg["class"], fold=f, epochs=EPOCHS, patience=PATIENCE, 
            lambda_boundary=cfg["lambda_boundary"], save_name=m_name
        )

        m_fold_metrics.append(best_res)
        m_fold_histories.append(hist)
        print(f"[{m_name}] Fold {f} Best: Dice = {best_res['dice']*100:.2f}%, IoU = {best_res['iou']*100:.2f}%, HD95 = {best_res['hd95']:.2f}")
        
    all_benchmark_results[m_name] = m_fold_metrics
    all_benchmark_histories[m_name] = m_fold_histories
    
    dices = [r["dice"] * 100 for r in m_fold_metrics]
    ious = [r["iou"] * 100 for r in m_fold_metrics]
    hd95s = [r["hd95"] for r in m_fold_metrics]
    print(f"\n--- {m_name.upper()} 5-Fold Summary ---")
    print(f"Per-Fold Dices: {[f'{d:.2f}%' for d in dices]}")
    print(f"Mean Dice : {np.mean(dices):.2f}% +/- {np.std(dices):.2f}%")
    print(f"Mean mIoU : {np.mean(ious):.2f}% +/- {np.std(ious):.2f}%")
    print(f"Mean HD95 : {np.mean(hd95s):.2f} +/- {np.std(hd95s):.2f} voxels")

# Save structured history JSON
with open("/kaggle/working/baselines_5fold_history.json", "w") as fp:
    json.dump({"results": all_benchmark_results, "histories": all_benchmark_histories}, fp, indent=2)

# Plot Convergence Dynamics
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)
colors = {"unet_plus_plus": "#d62728", "unet": "#1f77b4", "attention_unet": "#2ca02c"}

for m_name, hists in all_benchmark_histories.items():
    # Mean across 5 folds per epoch
    all_loss = np.array([h["train_loss"][:50] for h in hists if len(h["train_loss"])>=50])
    all_dice = np.array([h["val_dice"][:50] for h in hists if len(h["val_dice"])>=50])
    if len(all_loss) > 0:
        axes[0].plot(range(1, len(all_loss[0])+1), np.mean(all_loss, axis=0), label=m_name, color=colors.get(m_name, "black"), lw=2)
        axes[1].plot(range(1, len(all_dice[0])+1), np.mean(all_dice, axis=0), label=m_name, color=colors.get(m_name, "black"), lw=2)

axes[0].set_title("Baselines 5-Fold Mean Train Loss Convergence", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle=":", alpha=0.6)
axes[0].legend()

axes[1].set_title("Baselines 5-Fold Mean Validation Dice Convergence (%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice (%)")
axes[1].grid(True, linestyle=":", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/baselines_convergence_curve.png")
plt.close()

# Archive to ZIP
zip_out = "/kaggle/working/baselines_5fold_results.zip"
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("/kaggle/working/*.pth"): z.write(f, os.path.basename(f))
    for f in glob.glob("/kaggle/working/*.json"): z.write(f, os.path.basename(f))
    for f in glob.glob("/kaggle/working/*.png"): z.write(f, os.path.basename(f))
print(f"\n[COMPLETED] Successfully created archive: {zip_out} ({os.path.getsize(zip_out)/(1024*1024):.2f} MB)")
print("You can download 'baselines_5fold_results.zip' from Kaggle Output tab!")